# Outlier Detection & Handling

## Definition
**Outliers** are data points that differ significantly from the majority of observations.
They can:
- Distort statistical summaries (mean, variance)
- Degrade ML model performance (especially linear models, k-NN)
- Represent real phenomena (fraud detection) or errors (sensor malfunction)

### Detection Methods
| Method | Description |
|--------|-------------|
| **Z-score** | Values more than N std devs from mean |
| **IQR** | Values outside [Q1-1.5*IQR, Q3+1.5*IQR] |
| **Percentile clipping** | Cap values at Nth percentile |
| **Visual** | Boxplots, histograms |
| **Isolation Forest** | ML-based (sklearn) |

### Handling Strategies
- **Remove** — if errors or very small fraction
- **Cap/Clip** — Winsorization to a percentile
- **Transform** — log, sqrt to compress range
- **Bin** — convert to categorical

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 200
df = pd.DataFrame({
    'age':    rng.integers(20, 60, n).tolist() + [150, -5],   # outliers!
    'income': (rng.normal(60000, 15000, n).tolist() + [1_000_000, -5000]),
    'score':  rng.uniform(0, 100, n+2)
})
print(f'Dataset shape: {df.shape}')
print(df[['age', 'income']].describe())

In [ ]:
# Method 1: Z-score detection
from scipy import stats

z_age = np.abs(stats.zscore(df['age']))
outliers_z = df[z_age > 3]
print(f'Z-score outliers (|z| > 3): {len(outliers_z)}')
print(outliers_z[['age', 'income']])

In [ ]:
# Method 2: IQR method
def iqr_outliers(series, k=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - k * IQR
    upper = Q3 + k * IQR
    return (series < lower) | (series > upper), lower, upper

mask_age, lo, hi = iqr_outliers(df['age'])
print(f'IQR bounds for age: [{lo:.1f}, {hi:.1f}]')
print(f'Outliers detected: {mask_age.sum()}')
print(df[mask_age][['age', 'income']])

In [ ]:
# Method 3: Percentile clipping (Winsorization)
df['income_clipped'] = df['income'].clip(
    lower=df['income'].quantile(0.01),
    upper=df['income'].quantile(0.99)
)
print('Before clip max:', df['income'].max())
print('After clip max:', df['income_clipped'].max())

In [ ]:
# Handling: Remove outliers
mask_valid = ~(iqr_outliers(df['age'])[0] | iqr_outliers(df['income'])[0])
df_clean = df[mask_valid].copy()
print(f'After removing outliers: {df.shape} -> {df_clean.shape}')

In [ ]:
# Handling: Log transform (for right-skewed data like income)
df['income_log'] = np.log1p(np.abs(df['income_clipped']))   # log1p = log(1+x) handles 0
print('Income stats before log:')
print(df['income_clipped'].describe().round(0))
print('Income stats after log:')
print(df['income_log'].describe().round(3))

In [ ]:
# Column-wise outlier summary
def outlier_report(df, numeric_cols, k=1.5):
    for col in numeric_cols:
        mask, lo, hi = iqr_outliers(df[col], k)
        pct = 100 * mask.sum() / len(df)
        print(f'{col:15s}: {mask.sum():3d} outliers ({pct:.1f}%)  bounds=[{lo:.0f}, {hi:.0f}]')

outlier_report(df, ['age', 'income', 'score'])